Boolean algebra -> just inject x + negx = 1  and other equations. Then we don't have to more into xor-and ring.

one encoding is
clause1 -> true
clause2 -> true 

But actually that's just multiset not semiring? because clause is x \/ xneg
true \/ true -> true

Hyperresolution results from overlaps?

semiring boolean is richer but to whayt purppoe  
expr == true  is less comfortable than expr => p
p == expr could work with clark completion.  p = (\/ and and and). But even makeing that dnf would be a variable elimination? Well, but we're tseistin. We can use p elsewhere. and it doesn't have to be p -> body ordering (?) or that might not be possible if it isn't order like that


superposition centaur
resolution
KB
conditional KB (simpler than superposition?)

Injecting in star axioms or idempotency axioms of lattice/semilattice ought to give normalization procedure? How does this compare to resolution as semilattice reducer?
https://link.springer.com/chapter/10.1007/10721975_15 an algebra of resolution - struth

what are semiring proof objects
How to track proofs in any equational prover


p*_ + r as a mapping
p*_ + r : 0 -> 1   affine mappings out of 0 pick a polynomial?

multiterms ~ category
grobner et al should be viewed in cateorical light. Modules.
multiterms -> vectors?

trace knuth bendix


How could I have a stable base of solvers to work from? Well, why would I even want that. What is it all for?

Buchberger + binders

rig category. https://dl.acm.org/doi/10.5555/3089528.3089534 Computing with Semirings and Weak Rig Groupoids


Can egglog do it?

Just search for interesting normal forms

finite tropical semiring countermodels. That's interesting.

Candidates by using ring Z
x**2 + 1 = x implies x**7 = x also in ring

Mutually defined tree types?

Could we do an induction argument? 
(x*2 + c*x  + 1  -> x**7 = c*x )
&& 
x*2 + (c + 1)*x  + 1 
--> 
x**7 = (c+1)x   ?  

goal directed form. stop completion as soon as we hit our targetted equality.
Proof by saturation for something??? That could be fun.


Ok, these are kind of interesting benchmarks
bit tricks as benchmarks?
egglog people already have like giant neural networks as benchmarks

even just a counter of proof depth is kind of interesting
or emtting all deriving equations.

IS context and term the same thing
Contexts are 
a -> a homormophisms
gorund terms are 1 -> a  homormorphisms
systems of ground equations are 1 -> aaaa = 1 -> aaaa  
subterm is triangle
 find lhs in t ==>  t = ?ctx . lhs
t . subst1 = ?ctx . lhs . subst2



In [ ]:
%%file /tmp/termish.rs
#![allow(dead_code)]

use std::collections::HashMap;

trait Termish: Sized {
    type Ctx;

    fn plug(ctx: &Self::Ctx, value: &Self) -> Self;
    fn overlaps(&self, other: &Self) -> Vec<(Self::Ctx, Self::Ctx)>;
    fn find_one(&self, lhs: &Self) -> Option<Self::Ctx>;
}

enum Term {
    App(String, Vec<Term>),
    Var(String),
    Hole,
}

struct TermCtx {
    ctx: Term,
    subst: HashMap<String, Term>,
}

impl Termish for Term {
    type Ctx = TermCtx;

    fn plug(_ctx: &Self::Ctx, _value: &Self) -> Self { todo!() }
    fn overlaps(&self, _other: &Self) -> Vec<(Self::Ctx, Self::Ctx)> { todo!() }
    fn find_one(&self, _lhs: &Self) -> Option<Self::Ctx> { todo!() }
}

struct PairCtx<A, B>(A, B);

impl<T: Termish, S: Termish> Termish for (T, S) {
    type Ctx = PairCtx<T::Ctx, S::Ctx>;

    fn plug(ctx: &Self::Ctx, value: &Self) -> Self {
        (T::plug(&ctx.0, &value.0), S::plug(&ctx.1, &value.1))
    }

    fn overlaps(&self, _other: &Self) -> Vec<(Self::Ctx, Self::Ctx)> { todo!() }

    fn find_one(&self, lhs: &Self) -> Option<Self::Ctx> {
        Some(PairCtx(
            self.0.find_one(&lhs.0)?,
            self.1.find_one(&lhs.1)?,
        ))
    }
}

enum Either<T, S> {
    Left(T),
    Right(S),
}

enum EitherCtx<A, B> {
    Left(A),
    Right(B),
}

impl<T: Termish, S: Termish> Termish for Either<T, S> {
    type Ctx = EitherCtx<T::Ctx, S::Ctx>;

    fn plug(ctx: &Self::Ctx, value: &Self) -> Self {
        match (ctx, value) {
            (EitherCtx::Left(ctx), Either::Left(x)) => Either::Left(T::plug(ctx, x)),
            (EitherCtx::Right(ctx), Either::Right(x)) => Either::Right(S::plug(ctx, x)),
            _ => unreachable!(),
        }
    }

    fn overlaps(&self, _other: &Self) -> Vec<(Self::Ctx, Self::Ctx)> { todo!() }

    fn find_one(&self, lhs: &Self) -> Option<Self::Ctx> {
        match (self, lhs) {
            (Either::Left(x), Either::Left(y)) => x.find_one(y).map(EitherCtx::Left),
            (Either::Right(x), Either::Right(y)) => x.find_one(y).map(EitherCtx::Right),
            _ => None,
        }
    }
}

// Termish (A for F<A>) => Termish Fix(F)

struct Affine<T> {
    q: T,
    r: T,
}

struct Poly<T>(Vec<T>);
struct Semi<T>(Vec<T>);
struct MultiSet<T>(Vec<T>);

impl<T> Termish for Poly<T> {
    type Ctx = Affine<Self>; // q * _ + r

    fn plug(_ctx: &Self::Ctx, _value: &Self) -> Self { todo!() }
    fn overlaps(&self, _other: &Self) -> Vec<(Self::Ctx, Self::Ctx)> { todo!() }
    fn find_one(&self, _lhs: &Self) -> Option<Self::Ctx> { todo!() }
}

impl<T> Termish for Semi<T> {
    type Ctx = Affine<Self>; // q * _ + r

    fn plug(_ctx: &Self::Ctx, _value: &Self) -> Self { todo!() }
    fn overlaps(&self, _other: &Self) -> Vec<(Self::Ctx, Self::Ctx)> { todo!() }
    fn find_one(&self, _lhs: &Self) -> Option<Self::Ctx> { todo!() }
}

impl<T> Termish for MultiSet<T> {
    type Ctx = Self; // _ + r

    fn plug(_ctx: &Self::Ctx, _value: &Self) -> Self { todo!() }
    fn overlaps(&self, _other: &Self) -> Vec<(Self::Ctx, Self::Ctx)> { todo!() }
    fn find_one(&self, _lhs: &Self) -> Option<Self::Ctx> { todo!() }
}

struct Eq<T> {
    lhs: T,
    rhs: T,
}

struct Rewrite<T> {
    lhs: T,
    rhs: T,
}

fn rewrite_once<T: Termish>(value: &T, rw: &Rewrite<T>) -> Option<T> {
    let ctx = value.find_one(&rw.lhs)?;
    Some(T::plug(&ctx, &rw.rhs))
}

fn huet_complete<T>(_eqs: Vec<Eq<T>>) -> Vec<Rewrite<T>>
where
    T: Termish + PartialEq,
{
    todo!()
}

// Macaulay has an iterator interface which returns new rules.
// Goal completion could stop when the two goal normal forms agree.

fn main() {}

In [ ]:
! rustc /tmp/termish.rs -o /tmp/termish

In [ ]:

class Rewrite:
    lhs
    rhs
    pf :
 

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True, slots=True)
class Term:
    f : str
    args : tuple[Term, ...]

    def egglog(self):
        return f"{self.f} {" ".join([a.egglog() for a i nself.args])}
    





In [ ]:
%%file /tmp/seven.egg
(datatype Expr
  (Zero)
  (One)
  (X)
  (Add Expr Expr)
  (Mul Expr Expr))

(rewrite (Add (Zero) y) y)
(birewrite (Add x y) (Add y x))
(birewrite (Add x (Add y z)) (Add (Add x y) z))

(rewrite (Mul x (One)) x)
(rewrite (Mul x (Zero)) (Zero))
(birewrite (Mul x y) (Mul y x))
(birewrite (Mul x (Mul y z)) (Mul (Mul x y) z))

(birewrite (Mul x (Add y z))
           (Add (Mul x y) (Mul x z)))

(union (Add (One) (Mul (X) (X))) (X))

(let $x7 (Mul (X) (Mul (X) (Mul (X) (Mul (X) (Mul (X) (Mul (X) (X))))))))

;(run 20 :until (= $x7 (X)))
;(check (= $x7 (X)))
(run 5) ; at 5 it finishes
(extract $x7)

In [ ]:
%%file /tmp/leinster.egg
(datatype Expr
  (Zero)
  (One)
  (X)
  (Add Expr Expr)
  (Mul Expr Expr))

(rewrite (Add (Zero) y) y)
(birewrite (Add x y) (Add y x))
(birewrite (Add x (Add y z)) (Add (Add x y) z))

(rewrite (Mul x (One)) x)
(rewrite (Mul x (Zero)) (Zero))
(birewrite (Mul x y) (Mul y x))
(birewrite (Mul x (Mul y z)) (Mul (Mul x y) z))

(birewrite (Mul x (Add y z))
           (Add (Mul x y) (Mul x z)))

; 1 + x + x^2 = x
(union (Add (One) (Add (X) (Mul (X) (X)))) (X))

(let $x2 (Mul (X) (X)))
(let $x3 (Mul (X) $x2))
(let $x4 (Mul (X) $x3))
(let $x5 (Mul (X) $x4))

(let $r1 (Add (One) (Add (One) $x2)))
(let $l2 (Add (One) (Add $x2 $x3)))
(let $l3 (Add (One) (Add $x2 $x2)))
(let $l4 (Add (X) $x3))
(let $r4 (Add (One) $x2))

(run 20 :until (= $x5 (X)))

(check (= $x5 (X)))
(check (= $x4 $r1))
(check (= $l2 $x3))
(check (= $l3 $x2))
(check (= $l4 $r4))

(extract $x5)


In [67]:
%%file /tmp/seven.p

cnf(add_zero, axiom, add(zero,Y) = Y).
cnf(add_comm, axiom, add(X,Y) = add(Y,X)).
cnf(add_assoc, axiom, add(X,add(Y,Z)) = add(add(X,Y),Z)).

cnf(one_mul, axiom, mul(X,one) = X).
cnf(zero_mul, axiom, mul(X,zero) = zero).
cnf(mul_comm, axiom, mul(X,Y) = mul(Y,X)).
cnf(mul_assoc, axiom, mul(X,mul(Y,Z)) = mul(mul(X,Y),Z)).

cnf(distrib_left, axiom, mul(X,add(Y,Z)) = add(mul(X,Y),mul(X,Z))).


cnf(tree, axiom, add(one, mul(x,x)) = x). % x**2 + 1 = x
cnf(goal, axiom, myterm(mul(x,mul(x,mul(x,mul(x,mul(x,mul(x,x)))))))).
cnf(stump, axiom, true != false). % eprover doesn't need this

Overwriting /tmp/seven.p


In [61]:
! time eprover-ho --auto  --term-ordering=LPO4 --precedence="mul > add > one > zero > x" \
     --silent --generated-limit=70000 --print-saturated /tmp/seven.p | grep myterm

cnf(i_0_20, plain, (myterm(mul(x,mul(x,mul(x,mul(x,mul(x,mul(x,x))))))))).
cnf(i_0_60944, plain, (myterm(x))).

real	0m0.762s
user	0m0.716s
sys	0m0.063s


In [72]:
! twee --max-cps 100000 /tmp/seven.p  | grep myterm

  Axiom 10 (goal): myterm(mul(x, mul(x, mul(x, mul(x, mul(x, mul(x, x))))))) = true2.
10. myterm(mul(x, mul(x, mul(x, mul(x, mul(x, mul(x, x))))))) -> true2
  myterm(mul(x, mul(x, mul(x, mul(x, mul(x, mul(x, x))))))) -> true2


Works. doesn't hit very quickly. Oh that's interesting. It's tagged by clause number.


In [19]:
%%file /tmp/simp.p

cnf(add_zero, axiom, add(zero,Y) = Y).
cnf(add_comm, axiom, add(X,Y) = add(Y,X)).
cnf(add_assoc, axiom, add(X,add(Y,Z)) = add(add(X,Y),Z)).

cnf(add_neg, axiom, add(X,neg(X)) = zero).

cnf(one_mul, axiom, mul(X,one) = X).
cnf(zero_mul, axiom, mul(X,zero) = zero).
cnf(mul_comm, axiom, mul(X,Y) = mul(Y,X)).
cnf(mul_assoc, axiom, mul(X,mul(Y,Z)) = mul(mul(X,Y),Z)).

cnf(distrib_left, axiom, mul(X,add(Y,Z)) = add(mul(X,Y),mul(X,Z))).
cnf(myterm, axiom, myterm(mul(mul(x,one), zero))).
cnf(myterm2, axiom, myterm2(add(x,add(z, add(add(y,zero), neg(x)))))).


Overwriting /tmp/simp.p


In [20]:
! eprover-ho --auto --silent --generated-limit=10000 --print-saturated /tmp/simp.p | grep myterm

cnf(i_0_21, plain, (myterm(zero))).
cnf(i_0_22, plain, (myterm2(add(z,y)))).


In [82]:
%%file /tmp/integ.p

cnf(d_one, axiom, d(one) = zero).
cnf(d_zero, axiom, d(zero) = zero).
cnf(d_add, axiom, d(add(X,Y)) = add(d(X),d(Y))).
cnf(d_neg, axiom, d(neg(X)) = neg(d(X))).
cnf(d_mul, axiom, d(mul(X,Y)) = add(mul(d(X),Y),mul(X,d(Y)))).
cnf(add_zero, axiom, add(zero,Y) = Y).
cnf(add_comm, axiom, add(X,Y) = add(Y,X)).
cnf(add_assoc, axiom, add(X,add(Y,Z)) = add(add(X,Y),Z)).
cnf(add_neg, axiom, add(X,neg(X)) = zero).
cnf(one_mul, axiom, mul(X,one) = X).
cnf(zero_mul, axiom, mul(X,zero) = zero).
cnf(mul_comm, axiom, mul(X,Y) = mul(Y,X)).
cnf(mul_assoc, axiom, mul(X,mul(Y,Z)) = mul(mul(X,Y),Z)).

cnf(myterm, axiom, myterm(add(mul(y, d(x)), mul(x, d(y))))).

Overwriting /tmp/integ.p


In [84]:
! eprover-ho --auto --silent --generated-limit=10000  \
    --print-saturated /tmp/integ.p | grep myterm

cnf(i_0_28, plain, (myterm(d(mul(y,x))))).


In [40]:
%%file /tmp/integ.p

cnf(d_one, axiom, d(one) = zero).
cnf(d_zero, axiom, d(zero) = zero).
cnf(d_add, axiom, d(add(X,Y)) = add(d(X),d(Y))).
cnf(d_neg, axiom, d(neg(X)) = neg(d(X))).
cnf(d_mul, axiom, d(mul(X,Y)) = add(mul(d(X),Y),mul(X,d(Y)))).
cnf(d_sin, axiom, d(sin(X)) = mul(cos(X),d(X))).
cnf(d_cos, axiom, d(cos(X)) = neg(mul(cos(X),d(X)))).


cnf(add_zero, axiom, add(zero,Y) = Y).
cnf(add_comm, axiom, add(X,Y) = add(Y,X)).
cnf(add_assoc, axiom, add(X,add(Y,Z)) = add(add(X,Y),Z)).

cnf(add_neg, axiom, add(X,neg(X)) = zero).

cnf(one_mul, axiom, mul(X,one) = X).
cnf(zero_mul, axiom, mul(X,zero) = zero).
cnf(mul_x, axiom, mul(x,Y) = mul(Y,x)).
cnf(mul_y, axiom, mul(y,Y) = mul(Y,y)).
% mul_cos, mul_sin, ...
cnf(d_anti, axiom, mul(d(X),d(Y)) = neg(mul(d(Y),d(X)))).
% remove mul_comm if we want anticommute
%cnf(mul_comm, axiom, mul(X,Y) = mul(Y,X)).
cnf(mul_assoc, axiom, mul(X,mul(Y,Z)) = mul(mul(X,Y),Z)).

cnf(myterm, axiom, myterm(add(mul(y, d(x)), mul(x, d(y))))).



Overwriting /tmp/integ.p


In [41]:
! eprover-ho --auto --silent --generated-limit=100000  --print-saturated /tmp/integ.p | grep myterm

cnf(i_0_36, plain, (myterm(d(mul(x,y))))).


Putting `d` low on the ordering tends to pull it up.

In [31]:
! eprover-ho --auto --silent --generated-limit=10000 --term-ordering=LPO4  \
  --precedence="mul > add > neg > one > zero > d" --print-saturated /tmp/integ.p | grep myterm

cnf(i_0_32, plain, (myterm(d(mul(y,x))))).


# go bigger


            |
--------------|-----
x = 1 + x**2     |  x**7 = x | seven trees in one
x = 1 + x + x**2 | x^5 = x   | Leinster and fiore
x = 1 + 2x + x^2 | x^4 = x   | ??

x + n + x + x^2  | x^5 = n^2 x
x = c^2 + (c + 1)x + x^2 => x^4 = c^3 x
x = 2 + 3x + x**2 -> x^9 = 16x

These parametrized ones are 


Maybe this is kind of an interesting

x**3

| \((n,m,k)\) | Datatype equation | Smallest power |
|---|---|---|
| `(0,0,0)` | \(x=x^3\) | \(x^3=x\) |
| `(1,1,0)` | \(x=1+x+x^3\) | \(x^7=x\) |
| `(1,2,1)` | \(x=1+2x+x^2+x^3\) | \(x^5=x\) |
| `(1,3,2)` | \(x=1+3x+2x^2+x^3\) | \(x^7=x\) |

Ring thereoms that are not semiring theorems
| Equation | Ring predicts | Semiring status |
|---|---|---|
| \(x=2x+x^3\) | \(x^5=x\) | false |
| \(x=2x+x^2+x^3\) | \(x^4=x\) | false |


x = 1 + x + x³  =>  x⁷ = x
x = 2 + x + x³  =>  x⁷ = 4x
x = 3 + x + x³  =>  x⁷ = 9x
x = 4 + x + x³  =>  x⁷ = 16x
----
n + x + x**3 => x**7 = n**2 x ???

conjectures:
x = 8 + 5x + 2x² + x³  =>  x⁵ = 16x
x = 8 + 9x + 4x² + x³  =>  x⁷ = 64x

## Confirmed results and benchmark conjectures

Here `n` and `c` in the conjectures below mean positive natural-number coefficients (repeated sums of `1`), not new semiring generators.

### Confirmed semiring consequences

Checked with egglog and/or `semi.py` using only the commutative-semiring axioms and the displayed datatype equation.

| Datatype equation | Consequence |
|---|---|
| $x=1+x^2$ | $x^7=x$ |
| $x=1+x+x^2$ | $x^5=x$ |
| $x=1+2x+x^2$ | $x^4=x$ |
| $x=2+3x+x^2$ | $x^9=16x$ |
| $x=4+3x+x^2$ | $x^4=8x$ |
| $x=n+x+x^2$, checked separately for every $1\leq n\leq16$ | $x^5=n^2x$ |
| $x=c^2+(c+1)x+x^2$, for $c=1,2,3,4,5$ separately | $x^4=c^3x$ |
| $x=x^3$ | $x^3=x$ |
| $x=1+x+x^3$ | $x^7=x$ |
| $x=1+2x+x^2+x^3$ | $x^5=x$ |
| $x=1+3x+2x^2+x^3$ | $x^7=x$ |
| $x=n+x+x^3$, checked separately for every $1\leq n\leq6$ | $x^7=n^2x$ |
| $x=8+5x+2x^2+x^3$ | $x^5=16x$ |
| $cx=c^2+x^2$, with `c` an arbitrary generator | $cx^7=c^7x$ (18-step checked proof) |

### Conjectured positive-integer schemas

These are suggested by calculation in $\mathbb Z$ and by the confirmed instances, but still need uniform positive semiring proofs. They look useful as egglog / `semi.py` benchmarks.

$$x=n+x+x^2 \quad\Longrightarrow\quad x^5=n^2x \qquad(n\geq1)$$

$$x=c^2+(c+1)x+x^2 \quad\Longrightarrow\quad x^4=c^3x \qquad(c\geq1)$$

$$x=n+x+x^3 \quad\Longrightarrow\quad x^7=n^2x \qquad(n\geq1)$$

A bounded bidirectional search in `semi.py` proves the quadratic cases in under 0.4 seconds through $n=5$, the $c=3$ case in about 0.04 seconds, and the first cubic candidate above in about 0.18 seconds. Every returned path is checked step by step. The found path lengths are $n+11$, $c+9$, and $n+13$ for the three displayed families over the tested ranges, which looks like useful structure for uniform proofs. Full completion remains much harder. The second cubic candidate above is still a useful benchmark problem.

### The cyclotomic $\Phi_3$, $\Phi_4$, $\Phi_6$ pattern

$\Phi_r(z)$ is the $r$th cyclotomic polynomial: its roots are the primitive $r$th roots of unity. The three relevant quadratic polynomials are

$$\Phi_3(z)=z^2+z+1,\qquad \Phi_4(z)=z^2+1,\qquad \Phi_6(z)=z^2-z+1.$$

After ring cancellation, the three small datatype equations line up exactly with these:

| Cyclotomic polynomial | Datatype equation | Semiring consequence |
|---|---|---|
| $\Phi_3$ | $x=1+2x+x^2$ | $x^4=x$ |
| $\Phi_4$ | $x=1+x+x^2$ | $x^5=x$ |
| $\Phi_6$ | $x=1+x^2$ | $x^7=x$ |

So seven trees is part of the same cyclotomic pattern. Its useful homogeneous symbolic family has to retain the non-cancellable catalyst $c$:

$$cx=c^2+x^2 \quad\Longrightarrow\quad cx^7=c^7x.$$

The targeted `semi.py` search proves this with `c` as a genuine second generator in 18 checked steps (about 0.11 seconds). Setting $c=1$ recovers seven trees. Multiplying and chaining the base identity also gives the exponent family

$$cx=c^2+x^2 \quad\Longrightarrow\quad cx^{6k+1}=c^{6k+1}x \qquad(k\geq1),$$

and hence $x^{6k+1}=x$ when $c=1$. The tempting cancelled statement $x^7=c^6x$ is not the corresponding arbitrary-`c` semiring theorem.

### Important parameter distinction

If `n` or `c` is made into an arbitrary constant like `x`, the first two conjectured schemas are false. A finite truncated tropical semiring gives counterexamples to both. So those symbolic-constant versions are negative/model-finding benchmarks, not proof-search benchmarks. The homogeneous $\Phi_6$ statement above is different because it retains the necessary factor of $c$. The cases $n=0$ and $c=0$ also need to be excluded from the positive-integer schemas.

The ring calculation is only a candidate generator: cancellation gives $x^2=-n$ in the first family and $x^2+cx+c^2=0$ in the second, but those cancellation steps are not semiring-valid.

# multiset
manim animation?
priority queue of lcm might be useful

Hmm. Make a mip or milp.
explicit mpc
There is flex to remove rules if we could bound how much they hurt.


In [86]:
from collections import Counter

MS = Counter[str]
Rule = tuple[MS, MS]


def order(m: MS):
    """Degree, then lexicographic expanded-list order."""
    return m.total(), tuple(sorted(m.elements()))


def orient(x: MS, y: MS) -> Rule:
    return (x, y) if order(x) > order(y) else (y, x)


def step(x: MS, rule: Rule) -> MS | None:
    lhs, rhs = rule
    return x - lhs + rhs if lhs <= x else None


def nf(x: MS, rules: list[Rule]) -> MS:
    while True:
        for rule in rules:
            if (y := step(x, rule)) is not None:
                x = y
                break
        else:
            return x


def complete(equations: list[Rule]) -> list[Rule]:
    pending = equations.copy()
    rules: list[Rule] = []

    while pending:
        x, y = pending.pop()
        x, y = nf(x, rules), nf(y, rules)

        if x == y:
            continue

        lhs, rhs = orient(x, y)

        # Critical pairs with every previously inserted rule.
        for lhs1, rhs1 in rules:
            if lhs & lhs1:                 # nontrivial overlap
                overlap = lhs | lhs1       # componentwise maximum
                pending.append((
                    overlap - lhs  + rhs,
                    overlap - lhs1 + rhs1,
                ))

        rules.append((lhs, rhs))

    return rules

In [88]:
E = [
    (Counter(p=5),  Counter(n=1)),
    (Counter(p=10), Counter(d=1)),
    (Counter(p=25), Counter(q=1)),
]

R = complete(E)

assert nf(Counter(p=117), R) == Counter(
    p=2, n=1, d=1, q=4
)
R

[(Counter({'p': 25}), Counter({'q': 1})),
 (Counter({'p': 10}), Counter({'d': 1})),
 (Counter({'p': 5, 'd': 2}), Counter({'q': 1})),
 (Counter({'p': 5, 'q': 1}), Counter({'d': 3})),
 (Counter({'d': 5}), Counter({'q': 2})),
 (Counter({'p': 5}), Counter({'n': 1})),
 (Counter({'d': 3}), Counter({'q': 1, 'n': 1})),
 (Counter({'d': 2, 'q': 1, 'n': 1}), Counter({'q': 2})),
 (Counter({'q': 2, 'n': 2}), Counter({'q': 2, 'd': 1})),
 (Counter({'n': 2, 'q': 1}), Counter({'d': 1, 'q': 1})),
 (Counter({'d': 2, 'n': 1}), Counter({'q': 1})),
 (Counter({'n': 2}), Counter({'d': 1}))]

# embeddings
Yah, maybe this is dogshit.

In [ ]:
# An embedding e represents the context e[_].
from collections import Counter
from dataclasses import dataclass
from typing import Generic, Iterable, Protocol, TypeVar

T = TypeVar("T")
Embedding = TypeVar("Embedding")


@dataclass(frozen=True)
class Equation(Generic[T]):
    lhs: T
    rhs: T


@dataclass(frozen=True)
class Rewrite(Generic[T]):
    lhs: T
    rhs: T


@dataclass(frozen=True)
class Affine(Generic[T]):
    q: T
    r: T


class CompletionOps(Protocol[T, Embedding]):
    def orient(self, eq: Equation[T]) -> Rewrite[T] | None: ...

    # All e such that e[part] = whole.
    def embeddings(self, whole: T, part: T) -> Iterable[Embedding]: ...

    def plug(self, e: Embedding, value: T) -> T: ...

    # Minimal e1, e2 such that e1[left] = e2[right].
    def overlaps(self, left: T, right: T) -> Iterable[tuple[Embedding, Embedding]]: ...


def rewrite_once(value: T, rw: Rewrite[T], ops: CompletionOps[T, Embedding]):
    for e in ops.embeddings(value, rw.lhs):
        assert ops.plug(e, rw.lhs) == value
        result = ops.plug(e, rw.rhs)
        if result != value:
            yield result


def normal_form(
    value: T, rules: list[Rewrite[T]], ops: CompletionOps[T, Embedding]
) -> T:
    while True:
        for rw in rules:
            result = next(rewrite_once(value, rw, ops), None)
            if result is not None:
                value = result
                break
        else:
            return value


def critical_pairs(
    left: Rewrite[T], right: Rewrite[T], ops: CompletionOps[T, Embedding]
) -> Iterable[Equation[T]]:
    for e1, e2 in ops.overlaps(left.lhs, right.lhs):
        assert ops.plug(e1, left.lhs) == ops.plug(e2, right.lhs)
        yield Equation(ops.plug(e1, left.rhs), ops.plug(e2, right.rhs))


@dataclass
class _Marked(Generic[T]):
    rule: Rewrite[T]
    done: bool = False


def huet_complete(
    equations: list[Equation[T]],
    ops: CompletionOps[T, Embedding],
    *,
    goal: Equation[T] | None = None,
    max_rules: int = 100,
) -> list[Rewrite[T]]:
    pending = list(equations)
    entries: list[_Marked[T]] = []

    while True:
        while pending:
            rules = [entry.rule for entry in entries]
            eq = pending.pop()
            eq = Equation(
                normal_form(eq.lhs, rules, ops),
                normal_form(eq.rhs, rules, ops),
            )
            rw = ops.orient(eq)
            if rw is None or rw in rules:
                continue

            new_entries = [_Marked(rw)]
            for entry in entries:
                old = entry.rule
                lhs = normal_form(old.lhs, [rw], ops)
                if lhs == old.lhs:
                    rhs = normal_form(old.rhs, rules + [rw], ops)
                    if old.lhs != rhs:
                        new_entries.append(_Marked(Rewrite(old.lhs, rhs), entry.done))
                else:
                    pending.append(Equation(lhs, old.rhs))
            entries = new_entries

            rules = [entry.rule for entry in entries]
            if len(rules) > max_rules:
                raise RuntimeError("completion did not finish within max_rules")
            if goal is not None:
                if normal_form(goal.lhs, rules, ops) == normal_form(goal.rhs, rules, ops):
                    return rules

        entry = next((entry for entry in entries if not entry.done), None)
        if entry is None:
            return [entry.rule for entry in entries]
        entry.done = True
        rules = [entry.rule for entry in entries]
        for other in entries:
            for eq in critical_pairs(entry.rule, other.rule, ops):
                lhs = normal_form(eq.lhs, rules, ops)
                rhs = normal_form(eq.rhs, rules, ops)
                if lhs != rhs:
                    pending.append(Equation(lhs, rhs))

In [ ]:
class MultisetOps:
    @staticmethod
    def order(m: Counter):
        return m.total(), tuple(sorted(m.elements()))

    def orient(self, eq):
        if eq.lhs == eq.rhs:
            return None
        if self.order(eq.lhs) < self.order(eq.rhs):
            return Rewrite(eq.rhs, eq.lhs)
        return Rewrite(eq.lhs, eq.rhs)

    def embeddings(self, whole, part):
        if part <= whole:
            yield whole - part

    @staticmethod
    def plug(rest, value):
        return rest + value

    @staticmethod
    def overlaps(left, right):
        if left & right:
            overlap = left | right
            yield overlap - left, overlap - right


multiset_ops = MultisetOps()
multiset_rules = huet_complete([
    Equation(Counter(p=5), Counter(n=1)),
    Equation(Counter(p=10), Counter(d=1)),
    Equation(Counter(p=25), Counter(q=1)),
], multiset_ops)

assert normal_form(Counter(p=117), multiset_rules, multiset_ops) == Counter(
    p=2, n=1, d=1, q=4
)
multiset_rules

In [ ]:
from sympy import Poly, symbols


class BuchbergerOps:
    def __init__(self, *gens):
        self.gens = gens

    def poly(self, value=0):
        return Poly(value, *self.gens, domain="QQ")

    def term(self, monomial, coeff=1):
        return Poly.from_dict({monomial: coeff}, self.gens, domain="QQ")

    def orient(self, eq):
        difference = eq.lhs - eq.rhs
        if difference.is_zero:
            return None
        difference = difference.monic()
        lhs = self.term(difference.monoms()[0])
        return Rewrite(lhs, lhs - difference)

    def embeddings(self, whole, part):
        pm = part.monoms()[0]
        pc = part.coeff_monomial(pm)
        for monomial, coeff in whole.terms():
            if all(x >= y for x, y in zip(monomial, pm)):
                quotient = tuple(x - y for x, y in zip(monomial, pm))
                q = self.term(quotient, coeff / pc)
                yield Affine(q, whole - q * part)
                return

    @staticmethod
    def plug(e, value):
        return e.q * value + e.r

    def overlaps(self, left, right):
        lm, rm = left.monoms()[0], right.monoms()[0]
        lcm = tuple(max(x, y) for x, y in zip(lm, rm))
        lq = tuple(x - y for x, y in zip(lcm, lm))
        rq = tuple(x - y for x, y in zip(lcm, rm))
        zero = self.poly()
        yield Affine(self.term(lq), zero), Affine(self.term(rq), zero)


px, py = symbols("x y")
buchberger_ops = BuchbergerOps(px, py)
P = buchberger_ops.poly
buchberger_rules = huet_complete([
    Equation(P(px**2 - py), P()),
    Equation(P(px * py - 1), P()),
], buchberger_ops)

assert normal_form(P(py**3 - 1), buchberger_rules, buchberger_ops).is_zero
buchberger_rules

In [ ]:
try:
    from semi import Semi
except ModuleNotFoundError:
    from _drafts.semi import Semi


class SemiringOps:
    @staticmethod
    def orient(eq):
        if eq.lhs == eq.rhs:
            return None
        if eq.lhs < eq.rhs:
            return Rewrite(eq.rhs, eq.lhs)
        return Rewrite(eq.lhs, eq.rhs)

    @staticmethod
    def embeddings(whole, part):
        q, r = whole.divrem(part)
        if q.terms:
            yield Affine(q, r)

    @staticmethod
    def plug(e, value):
        return e.q * value + e.r

    @staticmethod
    def overlaps(left, right):
        for (q1, r1), (q2, r2) in left.overlaps_qr(right):
            yield Affine(q1, r1), Affine(q2, r2)


semiring_ops = SemiringOps()
sx = Semi.lit("x")
seven = Equation(sx**2 + 1, sx)
seven_goal = Equation(sx**7, sx)
semiring_rules = huet_complete([seven], semiring_ops, goal=seven_goal)

assert normal_form(seven_goal.lhs, semiring_rules, semiring_ops) == normal_form(
    seven_goal.rhs, semiring_rules, semiring_ops
)
semiring_rules